# L13 number representations — paper figure

Qwen2.5-7B-Instruct, `max(a, b)` one-shot prompt, residual stream leaving layer 13, at the last
token of each number. Loads `results/` written by `data_numberreps.py`.

Notation: $\mathbf{u}$ = DAS 1D trained at the **first-number** position (the first number at
its own location) · $\mathbf{v}_2$ = PC1 of the second-number cloud · $\mathbf{v}_1$ = the
DAS 1D direction inside head L14.H14's output, i.e. the first number *after transport* to the
second-number position. The two $\mathbf{v}$'s live in the same place, which is what
`makefig_sharedreps.ipynb` is about; $\mathbf{u}$ is the first number before it is moved.

The H14 panels read `results/sharedrep_*.npz` (written by `data_sharedrep.py`) — same model,
same prompt, same seed, so the two result files describe one run.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3d projection)

RESULTS_DIR, FIGS_DIR = 'results', 'figs'
os.makedirs(FIGS_DIR, exist_ok=True)

REGIMES = ['2digit', '3digit']
R    = {t: dict(np.load(f'{RESULTS_DIR}/numberreps_{t}.npz')) for t in REGIMES}
META = json.load(open(f'{RESULTS_DIR}/meta.json'))
MET  = {t: META['regimes'][t]['meta'] for t in REGIMES}

# The transport panels (the H14 direction, the per-head sweep) come from data_sharedrep.py.
# Only the regimes that have a sharedrep file are loaded, so this notebook still runs without it.
S, SMJ = {}, {}
_sp = f'{RESULTS_DIR}/sharedrep_meta.json'
for t in REGIMES:
    _np_ = f'{RESULTS_DIR}/sharedrep_{t}.npz'
    if os.path.exists(_np_) and os.path.exists(_sp):
        S[t] = dict(np.load(_np_))
        SMJ[t] = json.load(open(_sp))['regimes'][t]
print(f'sharedrep results loaded for: {sorted(S) or "(none)"}')

for t in REGIMES:
    m = MET[t]
    print(f"{t}: L{m['layer']}  numbers in [{m['low']}, {m['high']})  "
          f"cloud n={m['n_cloud']}  eval n={m['n_eval']}  "
          f"first-number tok {m['pos_first']}, second-number tok {m['pos_second']}")

## Plot style

Everything below reads these constants — change them here and re-run the figure cells.

In [ ]:
# Defaults for style_axes(); the paper-figure cell overrides each of them from FS.
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 20, 15, 16, 20
PANEL_FS = 24

# ---- Paper-figure font sizes -------------------------------------------------------------
# Every piece of text in the two paper figures reads from here, so there are no ad-hoc
# "LABEL_FS + 4" offsets buried in the figure cells. Edit one entry to change one thing, or
# bump FS_SCALE to grow them all together.
FS_SCALE = 1
FS = {k: v * FS_SCALE for k, v in dict(
    axis       = 20,   # x / y axis labels in panels (b) and (c)
    tick       = 16,   # 2D tick labels
    axis3d     = 17,   # PC axis labels in panel (a) -- kept lower, they collide easily
    tick3d     = 13,   # 3D tick labels
    cbar_label = 21,   # colorbar label
    cbar_tick  = 15,   # colorbar ticks
    legend     = 18,   # legend entries in (b)
    bar_label  = 18,   # 'full' / u / v2 / v1 above the bars in (e)
    group      = 26,   # the y1 / y2 group labels under (c)
    arrow      = 24,   # u beside the arrow in (a)
    panel      = 28,   # bold (a) / (b) / (c)
).items()}
DPI, SAVE = 300, True

# Sequential ramp for magnitude: perceptually uniform, CVD-optimized, and measured to sit
# >=10 OKLab dE from every series colour so panel (a) does not read as the series palette.
CMAP_VAL = 'cividis'
# Categorical series. Validated (OKLab dE, Machado-Oliveira-Fernandes CVD sim, WCAG):
#   CVD all-pairs 9.0 (target >=8) | normal-vision 17.5 (floor >=15) | min contrast 3.24 (>=3)
# The previous green/red pair FAILED at CVD dE 5.4 under deuteranopia. No red/green here.
COL = {'full': '#2166AC',   # blue
       'das':  '#C97B00',   # amber   -> u   (first number, at its own position)
       'pc1':  '#C06A9B',   # orchid  -> v2  (second number, at its own position)
       'h14':  '#117777'}   # teal    -> v1  (first number after transport, inside H14's output)
METHOD_NAME = {'full': 'full L13 residual', 'das': 'DAS 1D', 'pc1': 'PC1 1D'}
METHODS = ['full', 'das', 'pc1']
POS_COL = {0: COL['full'], 1: COL['pc1']}
POS_NAME = {0: 'first larger', 1: 'second larger'}
FIT_COL, FIT_LS = 'black', ':'
SCAT = dict(s=18, alpha=0.75, edgecolors='none')

# Which cloud / position each panel column refers to.
SLOTS = [('a', 'first number',  'A'),
         ('b', 'second number', 'B')]

# --- 3D PCA axis convention -------------------------------------------------------------------
# (x, y, z) = (PC2, PC3, PC1), so PC1 is the vertical axis and increases upward.
PC_ORDER = (1, 2, 0)
PC_NAMES = ('PC1', 'PC2', 'PC3')

# --- grey manifold trace ----------------------------------------------------------------------
CURVE_BINS, CURVE_COL, CURVE_LW = 14, '0.45', 2.2
CURVE_MS = 4.5
CURVE_SMOOTH = 1        # moving-average window over the binned means; 1 = no smoothing


def xyz_for_plot(coords):
    """Reorder (N, 3) PCA coordinates (or a (3,) direction) so PC1 lands on the vertical axis."""
    c = np.asarray(coords)
    return c[..., list(PC_ORDER)]


def manifold_curve(ax, coords, values, bins=CURVE_BINS, smooth=None, color=CURVE_COL, lw=CURVE_LW):
    """Grey trace through the cloud in order of increasing `values`: points are rank-sorted by
    value, split into `bins` equal-count groups, each group's mean position is taken, and the
    resulting polyline is moving-averaged with a window of `smooth` bins."""
    smooth = CURVE_SMOOTH if smooth is None else smooth
    order = np.argsort(values)
    c = xyz_for_plot(coords)[order]
    m = np.array([c[g].mean(axis=0) for g in np.array_split(np.arange(len(order)), bins) if len(g)])
    if smooth > 1:
        pad = np.pad(m, ((smooth // 2, smooth // 2), (0, 0)), mode='edge')
        k = np.ones(smooth) / smooth
        m = np.stack([np.convolve(pad[:, j], k, mode='valid')[:len(m)] for j in range(3)], axis=1)
    ax.plot(m[:, 0], m[:, 1], m[:, 2], '-o', color=color, lw=lw, ms=CURVE_MS,
            markerfacecolor='white', markeredgecolor=color, zorder=6)
    return m


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True,
               label_fs=None, tick_fs=None, title_fs=None):
    """The *_fs overrides let the paper-figure cells pass FS[...] without changing the
    exploratory sections, which keep the LABEL_FS / TICK_FS / TITLE_FS defaults."""
    label_fs = LABEL_FS if label_fs is None else label_fs
    tick_fs  = TICK_FS  if tick_fs  is None else tick_fs
    title_fs = TITLE_FS if title_fs is None else title_fs
    if xlabel: ax.set_xlabel(xlabel, fontsize=label_fs)
    if ylabel: ax.set_ylabel(ylabel, fontsize=label_fs)
    if title:  ax.set_title(title, fontsize=title_fs)
    ax.tick_params(labelsize=tick_fs)
    if grid: ax.grid(alpha=0.3)
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def style_axes3d(ax, title=None):
    ax.set_xlabel(PC_NAMES[PC_ORDER[0]], fontsize=LABEL_FS - 2)
    ax.set_ylabel(PC_NAMES[PC_ORDER[1]], fontsize=LABEL_FS - 2)
    ax.set_zlabel(PC_NAMES[PC_ORDER[2]], fontsize=LABEL_FS - 2)
    ax.tick_params(labelsize=TICK_FS - 3)
    if title: ax.set_title(title, fontsize=TITLE_FS - 2)


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/makefig_numberreps_{name}.png', dpi=DPI, bbox_inches='tight')

## Variance explained

Per-direction ratio for the top 3 PCs, and the cumulative spectrum, at each number position.

In [ ]:
fig, axes = plt.subplots(len(REGIMES), 2, figsize=(11, 4.2 * len(REGIMES)), squeeze=False)
for r, t in enumerate(REGIMES):
    d = R[t]
    ax = axes[r, 0]
    w, x = 0.38, np.arange(3)
    for i, (slot, posname, _v) in enumerate(SLOTS):
        ax.bar(x + (i - 0.5) * w, d[f'pca3_evr_{slot}'], w,
               color=COL['full'] if i == 0 else COL['pc1'], alpha=0.85, label=f'{posname} pos')
    ax.set_xticks(x); ax.set_xticklabels(['PC1', 'PC2', 'PC3'])
    style_axes(ax, ylabel='variance explained', title=f'{t} · top 3')
    ax.legend(fontsize=LEGEND_FS)

    ax = axes[r, 1]
    for i, (slot, posname, _v) in enumerate(SLOTS):
        s = d[f'pca_evr_spectrum_{slot}']
        ax.plot(np.arange(1, len(s) + 1), np.cumsum(s), 'o-', ms=4,
                color=COL['full'] if i == 0 else COL['pc1'], label=f'{posname} pos')
    ax.axhline(1.0, ls='--', color='0.6', lw=0.9)
    style_axes(ax, xlabel='number of components', ylabel='cumulative variance',
               title=f'{t} · spectrum')
    ax.legend(fontsize=LEGEND_FS)
plt.tight_layout()
savefig(fig, 'variance_explained')
plt.show()

## Paper figure — single-number representations, and the transport of the first one

**Row 1 — where each number is encoded, and what transport does to the first one's code.**
(a) PC1–PC3 PCA of the first number's cloud with $\mathbf{u}$ · (b) the first number's component along
$\mathbf{u}$ (at its own position) and along $\mathbf{v}_1$ (after L14.H14 copies it), on twin
axes · (c) the second number's component along $\mathbf{v}_2$.

**Row 2 — the head that does the transporting, and what each direction buys causally.**
(d) where H14's attention goes, and how it scales with $y_1$ · (e) which layer-14 head moves the
information · (f) IIA: every direction is causal for its own number, and the rank-1 direction is
more surgical than the full interchange.

In [ ]:
# ==================================================================================
# FONT SIZES  --  edit any of these
# ==================================================================================
FS_AXIS       = 24   # x / y axis labels, every panel
FS_TICK       = 20   # numeric tick labels, every panel
FS_PANEL      = 31   # the bold (a) ... (f)
FS_LEGEND     = 20   # legend entries
FS_ARROW      = 26   # (a) u beside the arrow
FS_CBAR_LABEL = 25   # (a) colorbar label
FS_CBAR_TICK  = 18   # (a) colorbar ticks
FS_INSET_AXIS = 17   # (d) inset axis labels
FS_INSET_TICK = 14   # (d) inset ticks
FS_BAR_LABEL  = 19   # (f) the direction name over each bar
FS_GROUP      = 24   # (f) the group labels under the bars

# ==================================================================================
# LAYOUT  --  edit any of these
# ==================================================================================
FIG_SIZE     = (22.5, 11.0)
# (b) carries axis labels on BOTH sides (twin axes), so the gap to (c) has to hold two of them --
# at a smaller wspace (b)'s right label and (c)'s left label print on top of each other.
GRID_WSPACE, GRID_HSPACE = 1.35, 0.36
# Six columns, two per panel.
WIDTH_RATIOS = (1.0, 1.0, 1.0, 1.0, 1.0, 1.0)
FIG_REGIME   = '2digit'
ARROW_SPAN, ARROW_LW, ARROW_HEAD, ARROW_LABEL_OFF = 0.38, 2.4, 18, 1.15
ARROW_COL    = '#111111'
CURVE_BINS_A = 12         # (a): the trace is the mean of this many equal-count bins of y1
CMAP_FIG     = 'plasma'   # min dE to the series colours: plasma 7.6 vs viridis 5.7
ATTN_INSET_BOX = [0.20, 0.46, 0.42, 0.48]        # (d): the profile's empty left half
TOP_N_HEADS  = 5          # (e): the rest of layer 14 is flat
PANEL_LABEL_X, PANEL_LABEL_Y = -0.15, 1.02
ALIGN_SIGNS  = True       # flip a component whose log fit slopes down -- signs are arbitrary

import matplotlib.patheffects as pe


d, mj = R[FIG_REGIME], META['regimes'][FIG_REGIME]
sd = S.get(FIG_REGIME)                     # H14 results; None until data_sharedrep.py has run
smj = SMJ.get(FIG_REGIME)
U, V1, V2 = r'$\mathbf{u}$', r'$\mathbf{v}_1$', r'$\mathbf{v}_2$'
Y1, Y2 = r'$y_1$', r'$y_2$'


def aggregate(x, y):
    """Group by the number value: (distinct y, mean f(y), SE). At each value the partner number
    ranges over the whole cloud, so this averages that nuisance variation out."""
    u = np.unique(x)
    m = np.array([y[x == v].mean() for v in u])
    se = np.array([y[x == v].std(ddof=1) / np.sqrt(max((x == v).sum(), 2)) for v in u])
    return u, m, se


def panel_letter(ax, s):
    ax.text(PANEL_LABEL_X, PANEL_LABEL_Y, s, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight='bold', ha='left', va='bottom')


def needs_sharedrep(ax):
    ax.text(0.5, 0.5, 'needs results/sharedrep_*.npz', ha='center', va='center',
            transform=ax.transAxes, fontsize=FS_AXIS, color='0.4')
    ax.set_axis_off()


fig = plt.figure(figsize=FIG_SIZE)
gs = fig.add_gridspec(2, 6, height_ratios=(1.0, 0.90), width_ratios=WIDTH_RATIOS,
                      wspace=GRID_WSPACE, hspace=GRID_HSPACE)

# ===================== (a) the first number's cloud, with u =====================================
ax = fig.add_subplot(gs[0, 0:2])
PCS_A = (0, 2)                                   # (a) shows PC1 vs PC3
coords, vals, evr = d['pca3_coords_a'][:, PCS_A], d['a_vals'], d['pca3_evr_a']
sc = ax.scatter(coords[:, 0], coords[:, 1], c=vals, cmap=CMAP_FIG, zorder=3, **SCAT)
# Trace: mean position per distinct y1, then averaged over equal-count bins of increasing y1.
_uv = np.unique(vals)
_path = np.array([coords[vals == v].mean(axis=0) for v in _uv])
trace = np.array([_path[b].mean(axis=0) for b in np.array_split(np.arange(len(_uv)), CURVE_BINS_A)])
ax.plot(trace[:, 0], trace[:, 1], '-o', color='0.2', lw=2.0, ms=3.5, zorder=5)
u2 = d['das_in_pca3_a'][list(PCS_A)]            # unit u's coefficients on the two PCs
if mj['logfit_das_a']['pearson_r'] < 0:          # point toward increasing y1
    u2 = -u2
u2_frac = np.linalg.norm(u2)                     # fraction of u lying in that plane
u2 = u2 / u2_frac
scale = ARROW_SPAN * np.ptp(coords @ u2)
tail = coords.mean(axis=0)                       # arrow starts at the cloud centre
ax.annotate('', xy=tail + u2 * scale, xytext=tail, zorder=12, annotation_clip=False,
            arrowprops=dict(arrowstyle='-|>', mutation_scale=ARROW_HEAD, lw=ARROW_LW,
                            color=ARROW_COL, shrinkA=0, shrinkB=0))
_t = ax.text(*(tail + u2 * scale * ARROW_LABEL_OFF), U, color=ARROW_COL, fontsize=FS_ARROW,
             ha='center', va='center', zorder=13)
_t.set_path_effects([pe.withStroke(linewidth=3, foreground='white')])
style_axes(ax, xlabel=f'PC{PCS_A[0] + 1} ({100 * evr[PCS_A[0]]:.0f}%)',
           ylabel=f'PC{PCS_A[1] + 1} ({100 * evr[PCS_A[1]]:.0f}%)',
           grid=False, label_fs=FS_AXIS, tick_fs=FS_TICK)
cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
cb.set_label(Y1, fontsize=FS_CBAR_LABEL); cb.ax.tick_params(labelsize=FS_CBAR_TICK)
cb.outline.set_visible(False)
panel_letter(ax, '(a)')
print(f"  u in the PC{PCS_A[0] + 1}-PC{PCS_A[1] + 1} plane: norm captured {u2_frac:.3f}")

# ============ (b) the first number, before and after transport, on twin axes ====================
# The same number read in two places: along u where it sits, and along v1 after H14 copies it.
# Twin axes rather than one scale -- the two components live in different spaces (the L13 residual
# and H14's output) and their units are not comparable; what is comparable is the SHAPE, and the
# shapes differ. Along u it is a clean log; along v1 it turns over at the bottom of the range and
# saturates at the top, so the copy is not faithful.
ax = fig.add_subplot(gs[0, 2:4])
blkU = mj['logfit_das_a']
sgnU = -1.0 if (ALIGN_SIGNS and blkU['p_all'] < 0) else 1.0
xu, mu, seu = aggregate(d['a_vals'], sgnU * d['das_comp_a'])
h1 = ax.errorbar(xu, mu, yerr=seu, fmt='o', ms=4.5, color=COL['das'], linestyle='none',
                 elinewidth=1.1, capsize=2, alpha=0.9, label=f'along {U}')
xf = np.linspace(d['a_vals'].min(), d['a_vals'].max(), 300)
ax.plot(xf, sgnU * (blkU['p_all'] * np.log(xf) + blkU['q_all']), '--', color=COL['das'], lw=2.2)
style_axes(ax, xlabel=Y1, ylabel=f"$f_1'(y_1)$, comp. along {U}", grid=False,
           label_fs=FS_AXIS, tick_fs=FS_TICK)
ax.yaxis.label.set_color(COL['das']); ax.tick_params(axis='y', colors=COL['das'])
ax.spines['left'].set_color(COL['das'])
handles = [h1]
if sd is None:
    ax.legend(handles=handles, fontsize=FS_LEGEND, frameon=False)
else:
    ax2 = ax.twinx()
    blkV = smj['logfit_h14_das_a']
    sgnV = -1.0 if blkV['pearson_r'] < 0 else 1.0
    xv, mv, sev = aggregate(sd['a_vals'], sgnV * sd['h14_das_comp'].astype(float))
    ax2.plot(xv, mv, '-', color=COL['h14'], lw=1.0, alpha=0.45, zorder=1)
    h2 = ax2.errorbar(xv, mv, yerr=sev, fmt='s', ms=4.0, color=COL['h14'], linestyle='none',
                      elinewidth=1.0, capsize=2, alpha=0.95, zorder=2,
                      label=f'along {V1}')
    ax2.set_ylabel(f'$f_1(y_1)$, comp. along {V1}', fontsize=FS_AXIS, color=COL['h14'])
    ax2.tick_params(labelsize=FS_TICK, colors=COL['h14'])
    ax2.spines['right'].set_color(COL['h14']); ax2.spines['top'].set_visible(False)
    ax2.grid(False)
    ax.legend(handles=[h1, h2], fontsize=FS_LEGEND, frameon=False, loc='upper left',
              borderaxespad=0.2, handletextpad=0.3)
panel_letter(ax, '(b)')

# ===================== (c) the second number, at its own position ===============================
ax = fig.add_subplot(gs[0, 4:6])
blkW = mj['logfit_pc1_b']
sgnW = -1.0 if (ALIGN_SIGNS and blkW['p_all'] < 0) else 1.0
xw, mw, sew = aggregate(d['b_vals'], sgnW * d['pca3_coords_b'][:, 0])
ax.errorbar(xw, mw, yerr=sew, fmt='o', ms=4.5, color=COL['pc1'], linestyle='none',
            elinewidth=1.1, capsize=2, alpha=0.9)
xf = np.linspace(d['b_vals'].min(), d['b_vals'].max(), 300)
ax.plot(xf, sgnW * (blkW['p_all'] * np.log(xf) + blkW['q_all']), '--', color=COL['pc1'], lw=2.2)
style_axes(ax, xlabel=Y2, ylabel=f'$f_2(y_2)$, comp. along {V2}', grid=False,
           label_fs=FS_AXIS, tick_fs=FS_TICK)
panel_letter(ax, '(c)')

# ===================== (d) what the head reads ==================================================
ax = fig.add_subplot(gs[1, 0:2])
if sd is None or 'h14_attn_row_n2' not in sd:
    needs_sharedrep(ax)
else:
    row = sd['h14_attn_row_n2'].astype(float)
    mA, sA = row.mean(axis=0), row.std(axis=0) / np.sqrt(row.shape[0])
    tpos = np.arange(row.shape[1])
    _m = smj['meta']
    lo_ = _m['pos_first'] - _m['n_tok_per_number'][0] + 1
    ax.axvspan(lo_ - 0.5, _m['pos_first'] + 0.5, color=COL['das'], alpha=0.14, zorder=0)
    ax.errorbar(tpos, mA, yerr=sA, fmt='-o', ms=3.0, lw=1.2, color=COL['das'], elinewidth=1.0,
                capsize=1.5, zorder=2)
    style_axes(ax, xlabel='token position', ylabel='H14 attention',
               grid=False, label_fs=FS_AXIS, tick_fs=FS_TICK)
    # Only the first number's position is labelled; the rest are ticks, to show the axis is over
    # tokens. The band is that number's whole token span.
    ax.set_xticks(tpos)
    ax.set_xticklabels([f'{Y1}\nposition' if t == _m['pos_first'] else '' for t in tpos],
                       fontsize=FS_AXIS)
    ax.tick_params(axis='x', length=3, pad=2)
    ax.margins(x=0.02)
    # How much the head attends is itself a function of the number, so part of (b)'s curve is the
    # attention and not only what is copied. Inset rather than a caption sentence.
    axi = ax.inset_axes(ATTN_INSET_BOX)
    xa, ma, sa = aggregate(sd['a_vals'], sd['h14_attn_to_n1'].astype(float))
    axi.errorbar(xa, ma, yerr=sa, fmt='o', ms=2.6, color=COL['das'], linestyle='none',
                 elinewidth=0.8, alpha=0.9)
    axi.set_xlabel(Y1, fontsize=FS_INSET_AXIS)
    axi.set_ylabel(f'attention to {Y1}', fontsize=FS_INSET_AXIS)
    axi.tick_params(labelsize=FS_INSET_TICK)
    axi.set_facecolor('white'); axi.patch.set_alpha(1.0); axi.set_zorder(5)
    for _s in ('top', 'right'):
        axi.spines[_s].set_visible(False)
    print(f"  H14 attention: {sd['h14_attn_to_n1'].mean():.3f} on {Y1}, "
          f"corr with y1 {np.corrcoef(sd['a_vals'], sd['h14_attn_to_n1'])[0, 1]:+.3f}")
panel_letter(ax, '(d)')

# ===================== (e) which head does the transport ========================================
# Each layer-14 head's own output contribution interchanged where the second number sits, alone
# and jointly with an L13 v2 interchange at the same place. The joint arm is the one that matters,
# because H14 writes on top of whatever L13 already holds there; the dashed line is the L13 patch's
# own effect, so a bar reads as what the head adds on top of it.
ax = fig.add_subplot(gs[1, 2:4])
if sd is None:
    needs_sharedrep(ax)
else:
    ARMS_N = [('alone', 'head alone', COL['full']),
              ('with_L13_PC1_n2', f'head & {V2}', COL['das'])]
    order = np.argsort(-sd['head_posrec_with_L13_PC1_n2'].mean(axis=1))
    sel = order[:TOP_N_HEADS]
    xh, Wb = np.arange(TOP_N_HEADS), 0.8 / len(ARMS_N)
    for ai, (arm, lab, col) in enumerate(ARMS_N):
        v = sd[f'head_posrec_{arm}'][sel]
        ax.bar(xh + (ai - (len(ARMS_N) - 1) / 2) * Wb, v.mean(axis=1), Wb,
               yerr=v.std(axis=1) / np.sqrt(v.shape[1]), capsize=3, error_kw=dict(lw=1.0),
               color=col, alpha=0.9, label=lab)
    ax.axhline(sd['cond_posrec_L13_PC1_atn2'].mean(), ls='--', color=COL['das'], lw=1.4,
               alpha=0.8, zorder=1, label=f'{V2} alone')
    ax.axhline(0, color='0.6', lw=0.9)
    ax.margins(y=0.22)
    ax.set_xticks(xh); ax.set_xticklabels([f'H{h}' for h in sel], fontsize=FS_TICK)
    ax.set_xlim(-0.6, TOP_N_HEADS - 0.4)
    style_axes(ax, xlabel=f"layer {smj['meta']['h_layer']} heads "
                          f"(top {TOP_N_HEADS} of {smj['meta']['n_heads']})",
               ylabel='PR (pos. recovery)', grid=False,
               label_fs=FS_AXIS, tick_fs=FS_TICK)
    ax.legend(fontsize=FS_LEGEND - 2, frameon=False, loc='upper right')
panel_letter(ax, '(e)')

# ===================== (f) IIA: each direction is causal for its own number =====================
# One group per number-and-place; within a group, the full interchange at that scope against the
# rank-1 direction. In two of the three the rank-1 direction scores far HIGHER than the full swap:
# a full interchange drags in everything else at that site, the learned direction is the surgical
# version of the same intervention.
ax = fig.add_subplot(gs[1, 4:6])
# v1 sits in the y1 group, not in a group of its own: it is the same number, read after H14 has
# copied it. Its own full-scope comparator (interchanging H14's entire output contribution) scores
# 0.00 and is printed rather than drawn -- a second blue "full" bar inside one group, at a
# different scope from the first, reads as the same control twice.
GROUPS = [(Y1, [('full', d['iia_full_n1_larger_first_number'], COL['full']),
                (U,      d['iia_das_n1_larger_first_number'],  COL['das'])]),
          (Y2, [('full', d['iia_full_n2_larger_second_number'], COL['full']),
                (V2,     d['iia_pc1b_n2_larger_second_number'], COL['pc1'])])]
if sd is not None:
    GROUPS[0][1].append((V1, sd['cond_iia_H14_DAS'], COL['h14']))
    print(f"  H14 whole-output interchange (not drawn): "
          f"{sd['head_iia_alone'][smj['meta']['h_head']].mean():.3f}")
W = 0.28
for gi, (glab, bars) in enumerate(GROUPS):
    for bi, (blab, v, col) in enumerate(bars):
        xpos = gi + (bi - (len(bars) - 1) / 2) * W
        se = v.std() / np.sqrt(len(v))
        ax.bar(xpos, v.mean(), W, yerr=se, capsize=4, color=col, alpha=0.9)
        ax.text(xpos, v.mean() + se + 0.03, f'{blab}\n{v.mean():.2f}', ha='center', va='bottom',
                fontsize=FS_BAR_LABEL, linespacing=1.25)
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels([g[0] for g in GROUPS], fontsize=FS_GROUP)
ax.set_ylim(0, 1.30)
style_axes(ax, xlabel='number perturbed', ylabel='IIA', grid=False, label_fs=FS_AXIS,
           tick_fs=FS_TICK)
ax.tick_params(axis='x', labelsize=FS_GROUP)      # style_axes just reset it to FS_TICK
panel_letter(ax, '(f)')

fig.subplots_adjust(left=0.045, right=0.985, top=0.95, bottom=0.07)
savefig(fig, 'number_reps_fig')
# plt.savefig('./figs_mainpaper/number_reps_fig.pdf', bbox_inches='tight', dpi=300)
plt.show()

## The transported number's manifold

H14's own output cloud in its top two PCs, coloured by $y_1$, with the mean position per number
(smoothed) drawn through it. Supports the claim panel (b) makes numerically: the transported code
is a curved manifold, and $\mathbf{v}_1$ is one straight chord through it.

In [ ]:
MAN_FS = dict(axis=17, tick=14, cbar=17, title=17)
MAN_SMOOTH = 9        # moving average over the per-number means; the raw path is jittery
MAN_REGIME = FIG_REGIME

_s = S.get(MAN_REGIME)
if _s is None or 'h14_pca_coords' not in _s:
    print('needs results/sharedrep_*.npz')
else:
    hp, hevr, av = _s['h14_pca_coords'][:, :2].astype(float), _s['h14_pca_evr'], _s['a_vals']
    uq = np.unique(av)
    path = np.array([hp[av == v].mean(axis=0) for v in uq])
    k = MAN_SMOOTH
    pad = np.pad(path, ((k // 2, k // 2), (0, 0)), mode='edge')
    path = np.stack([np.convolve(pad[:, j], np.ones(k) / k, mode='valid')[:len(uq)]
                     for j in range(2)], axis=1)

    fig, ax = plt.subplots(figsize=(6.6, 5.6))
    sc = ax.scatter(hp[:, 0], hp[:, 1], c=av, cmap=CMAP_FIG, s=8, alpha=0.55, edgecolors='none')
    ax.plot(path[:, 0], path[:, 1], '-', color='0.2', lw=2.2, zorder=5)
    style_axes(ax, xlabel=f'PC1 ({100 * hevr[0]:.0f}%)', ylabel=f'PC2 ({100 * hevr[1]:.0f}%)',
               label_fs=MAN_FS['axis'], tick_fs=MAN_FS['tick'])
    cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label(Y1, fontsize=MAN_FS['cbar']); cb.ax.tick_params(labelsize=MAN_FS['tick'])
    cb.outline.set_visible(False)

    # How much of the trajectory the straight direction accounts for. Arc length is computed on
    # the smoothed path: the raw per-number path jitters, which inflates it.
    dv = _s['h14_das_dir'].astype(float); dv /= np.linalg.norm(dv)
    full_path = np.array([(_s['h14_cloud'].astype(float) -
                           _s['h14_cloud'].astype(float).mean(0))[av == v].mean(0) for v in uq])
    padf = np.pad(full_path, ((k // 2, k // 2), (0, 0)), mode='edge')
    smf = np.stack([np.convolve(padf[:, j], np.ones(k) / k, mode='valid')[:len(uq)]
                    for j in range(full_path.shape[1])], axis=1)
    step = np.diff(smf, axis=0)
    net = smf[-1] - smf[0]
    print(f'top-2 PCs hold {100 * hevr[:2].sum():.0f}% of the head output variance')
    print(f'along v1: {100 * abs(net @ dv) / np.linalg.norm(net):.0f}% of the net displacement, '
          f'{100 * np.abs(step @ dv).sum() / np.linalg.norm(step, axis=1).sum():.0f}% of the arc '
          f'length -- and that chord alone gives IIA '
          f'{_s["cond_iia_H14_DAS"].mean():.2f}')
    print('caveat: an arc in PC1-PC2 is also the horseshoe artifact of PCA on a 1-D gradient; the '
          'independent evidence is the non-monotone v1 curve in panel (b), measured in the full '
          'space.')
    plt.tight_layout(); savefig(fig, 'h14_output_manifold'); plt.show()